# Create HHS TAGGS Awards (multi-funder: SAMHSA, IHS, CMS, ASPR, HHS/OS)

Creates awards from **HHS TAGGS** (Tracking Accountability in Government Grants
System, https://taggs.hhs.gov) — HHS's own first-party grants register covering
all HHS operating divisions (OPDIVs).

**Prerequisites:**
- Run `scripts/local/hhs_taggs_to_s3.py` to harvest and upload the data first.

**Data source:** https://taggs.hhs.gov/SearchAward (session-based CSV export)
**S3 location:** `s3a://openalex-ingest/awards/hhs_taggs/hhs_taggs_projects.parquet`
**Provenance:** `hhs_taggs` | **Priority:** 417

## Multi-funder attribution (runbook §2.3.2)

TAGGS is a shared reporting system bundling MANY funders — each OPDIV's grants
are attributed to its own OpenAlex funder entity, never blanket-assigned:

| OPDIV | OpenAlex funder | funder_id | path |
|-------|-----------------|-----------|------|
| SAMHSA | Substance Abuse and Mental Health Services Administration | 4320332164 | A (dim) |
| IHS | Indian Health Service | 4320332207 | A (dim) |
| CMS | Centers for Medicare and Medicaid Services | 4320332205 | A (dim) |
| DHHS/OS | U.S. Department of Health and Human Services (umbrella) | 4320306085 | A (dim) |
| ASPR | Administration for Strategic Preparedness and Response | 1724715131 | **B (inline — non-F4320\*)** |

DHHS/OS = Office of the Secretary staff divisions (OASH, OMH, ONC, OPA…);
no OPDIV-level OpenAlex funder exists, so its competitive discretionary program
grants (incl. Research on Research Integrity, LEAP health IT, vaccine-safety
research, minority-health and Title X programs) map to the HHS umbrella funder.

**OPDIVs deliberately NOT in this parquet** (avoid dedup noise with existing
ingests): NIH (Complete first-party via ExPORTER), CDC (usaspending_cdc p55),
AHRQ (usaspending_ahrq p54), HRSA (hrsa data warehouse p57), ACF
(usaspending_acf p233), ACL (usaspending_acl p235). FDA rows are staged
separately at `s3a://openalex-ingest/awards/hhs_taggs/staging_fda.parquet`
(an FDA build is in flight elsewhere) and are **not** read by this notebook.

## Inclusion rule (research/discretionary scope)

TAGGS includes huge non-research service/entitlement assistance. We keep only
awards whose **Award Class Type is `DISCRETIONARY` or `COOPERATIVE AGREEMENT`**
(competitively awarded, acknowledgeable assistance) and drop `BLOCK`,
`OPEN-ENDED`, `CLOSED-ENDED`, `DIRECT PAYMENT FOR SPECIFIED USE` (block grants,
Medicaid-adjacent open-ended entitlements, tribal direct payments). Empirically
each award carries exactly one class type. Expected: ~13.8K of ~20.2K awards
survive the scope filter.

## Source semantics (verified 2026-07-12)

- TAGGS search exports only the **most recent 5 fiscal years** (FY2022–FY2026).
- Parquet rows are **obligation actions** (one award = many actions); this
  notebook aggregates to award level by `(opdiv, award_number)`.
- `amount` = SUM of signed obligation actions in the FY2022–26 window
  (parenthesized amounts are de-obligations, i.e. negative). Sums ≤ 0 (e.g.
  continuation awards with only $0 admin actions in-window) ship as NULL —
  expect ~62% non-null amount coverage. Currency: USD (implicit, US federal).
- `start_date` = the action date of the award's **budget period 1** action when
  observed in-window (i.e. the award actually started FY2022+); otherwise NULL
  rather than fabricated — expect ~64% coverage. `end_date` is not published.
- PI names are **not published** in the export → `lead_investigator` NULL by
  design; recipient org kept in helper columns (`recipient_name`, `recipient_state`).
- `landing_page_url` NULL: TAGGS detail URLs need a ProgOfficeCode the export
  doesn't include.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet (obligation-action level)
CREATE OR REPLACE TABLE openalex.awards.hhs_taggs_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hhs_taggs/hhs_taggs_projects.parquet`;

In [ ]:
%sql
-- Check row count (expect 117,816 obligation actions; FY2022-2026;
-- OPDIVs SAMHSA/IHS/CMS/ASPR/DHHS-OS)
SELECT COUNT(*) as total_actions FROM openalex.awards.hhs_taggs_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT
    issue_fy, fund_fy, opdiv, aln, assistance_listing, state,
    award_number, award_title, award_code, award_class_type,
    budget_year, action_date, legal_entity_name, award_amount
FROM openalex.awards.hhs_taggs_raw
LIMIT 5;

## Step 1.5: Money-field scan + raw inspection

The export's only money column is `award_amount` (string like `$1,234,567`,
parenthesized when a de-obligation). Run the runbook §1.5 scan anyway to
confirm nothing money-shaped is missed.

In [ ]:
%sql
-- Money-field scan per runbook §1.5
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'hhs_taggs_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant_offer|awarded';

In [ ]:
%sql
-- Sanity-check the parsed amount distribution (action level).
-- Real grant obligations: thousands..millions. ~44% of actions are $0
-- (admin actions) and ~12% negative (de-obligations) — both expected.
SELECT
    MIN(amt) AS min_val, MAX(amt) AS max_val, AVG(amt) AS avg_val,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN amt = 0 THEN 1 ELSE 0 END) AS zero_actions,
    SUM(CASE WHEN amt < 0 THEN 1 ELSE 0 END) AS negative_actions
FROM (
    SELECT TRY_CAST(REGEXP_REPLACE(award_amount, '[$,()]', '') AS DOUBLE)
             * CASE WHEN award_amount LIKE '%(%' THEN -1 ELSE 1 END AS amt
    FROM openalex.awards.hhs_taggs_raw
);

## Step 1.6: Fail-fast — verify the funder rows exist

Four of the five target funders are F4320\* and MUST be present in
`openalex.common.funder` (Path A). ASPR (F1724715131) is non-F4320\* so it is
**expected to be absent** from the dim (Path B) — its canonical values are
inlined in the Step 2 funder_map from https://api.openalex.org/funders/F1724715131.

In [ ]:
%sql
-- Must return exactly 4 rows (SAMHSA, IHS, CMS, HHS umbrella).
-- If fewer, STOP — a Path A funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id IN (4320332164, 4320332207, 4320332205, 4320306085)
ORDER BY funder_id;

## Step 2: Create HHS TAGGS Awards Table

Aggregates obligation actions to award level, applies the
research/discretionary scope filter, and attributes each OPDIV to its funder.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hhs_taggs_awards
USING delta
AS
WITH
-- OPDIV -> OpenAlex funder map (runbook §2.3.2: never blanket-assign one funder).
-- Path A rows come from openalex.common.funder; ASPR is non-F4320* (Path B) so
-- its canonical values are inlined from https://api.openalex.org/funders/F1724715131.
funder_map AS (
    SELECT m.opdiv, f.funder_id, f.display_name, f.ror_id, f.doi
    FROM (VALUES
        ('SAMHSA',  4320332164),
        ('IHS',     4320332207),
        ('CMS',     4320332205),
        ('DHHS/OS', 4320306085)
    ) AS m(opdiv, funder_id)
    JOIN openalex.common.funder f ON f.funder_id = m.funder_id
    UNION ALL
    SELECT
        'ASPR',
        1724715131,
        'Administration for Strategic Preparedness and Response',
        'https://ror.org/05tjhqa05',
        '10.13039/100021704'
),

-- Parse action-level fields defensively
actions AS (
    SELECT
        opdiv,
        UPPER(TRIM(award_number)) AS award_number,
        NULLIF(TRIM(award_title), '') AS award_title,
        NULLIF(TRIM(aln), '') AS aln,
        NULLIF(TRIM(assistance_listing), '') AS assistance_listing,
        NULLIF(TRIM(award_class_type), '') AS award_class_type,
        NULLIF(TRIM(state), '') AS state,
        NULLIF(TRIM(legal_entity_name), '') AS legal_entity_name,
        TRY_CAST(REGEXP_REPLACE(award_amount, '[$,()]', '') AS DOUBLE)
          * CASE WHEN award_amount LIKE '%(%' THEN -1 ELSE 1 END AS amount_signed,
        COALESCE(
            TRY_TO_DATE(action_date, 'M/d/yyyy'),
            TRY_TO_DATE(action_date, 'MM/dd/yyyy')
        ) AS action_dt,
        TRY_CAST(budget_year AS INT) AS budget_yr
    FROM openalex.awards.hhs_taggs_raw
    WHERE award_number IS NOT NULL AND TRIM(award_number) != ''
),

-- Roll obligation actions up to award level
awards_rolled AS (
    SELECT
        opdiv,
        award_number,
        MAX_BY(award_title, LENGTH(award_title)) AS award_title,
        MODE(assistance_listing) AS assistance_listing,
        MODE(award_class_type) AS award_class_type,
        MODE(legal_entity_name) AS recipient_name,
        MODE(state) AS recipient_state,
        SUM(amount_signed) AS total_obligated,
        -- start only when the award's budget-period-1 action is in-window;
        -- otherwise NULL (the true start predates the FY2022-26 export window)
        MIN(CASE WHEN budget_yr = 1 THEN action_dt END) AS start_dt,
        MIN(action_dt) AS first_action_dt,
        MAX(action_dt) AS last_action_dt,
        COUNT(*) AS n_actions
    FROM actions
    GROUP BY opdiv, award_number
),

-- Research/discretionary scope (see header): competitively awarded classes only
awards_scoped AS (
    SELECT * FROM awards_rolled
    WHERE award_class_type IN ('DISCRETIONARY', 'COOPERATIVE AGREEMENT')
),

awards_transformed AS (
    SELECT
        -- Unique ID from funder_id : native award number
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(a.award_number)))) % 9000000000 as id,

        a.award_title as display_name,

        -- TAGGS export has no abstract/description
        CAST(NULL AS STRING) as description,

        f.funder_id,
        a.award_number as funder_award_id,

        -- Sum of signed obligations in the FY2022-26 window; NULL when <= 0
        CASE WHEN a.total_obligated > 0 THEN a.total_obligated END as amount,
        'USD' as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'grant' as funding_type,

        -- Scheme = Assistance Listing (CFDA) program name
        a.assistance_listing as funder_scheme,

        'hhs_taggs' as provenance,

        a.start_dt as start_date,
        CAST(NULL AS DATE) as end_date,
        YEAR(a.start_dt) as start_year,
        CAST(NULL AS INT) as end_year,

        -- PI not published in the TAGGS search export -> NULL by design
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- TAGGS detail pages need a ProgOfficeCode the export doesn't carry
        CAST(NULL AS STRING) as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(a.award_number)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date,

        -- Helper columns (not inserted into openalex_awards_raw)
        a.opdiv,
        a.award_class_type,
        a.recipient_name,
        a.recipient_state,
        a.n_actions,
        a.first_action_dt,
        a.last_action_dt

    FROM awards_scoped a
    JOIN funder_map f ON f.opdiv = a.opdiv
)

SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hhs_taggs' AND priority = 417;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    417 as priority  -- HHS TAGGS priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.hhs_taggs_awards;

## Verification Queries

In [ ]:
%sql
-- Check row count (expect ~13,831 in-scope awards from ~20,214 rolled awards)
SELECT COUNT(*) as total_hhs_taggs_awards FROM openalex.awards.hhs_taggs_awards;

In [ ]:
%sql
-- Sample the data
SELECT
    id, display_name, funder_award_id, funder_scheme, funding_type,
    amount, start_date, opdiv, award_class_type, recipient_name
FROM openalex.awards.hhs_taggs_awards
LIMIT 10;

In [ ]:
%sql
-- §6.5 Funder consistency: the §2.3.2 anti-blanket check.
-- Expect FIVE funders with plausible splits (from the harvest profile):
-- SAMHSA ~10,179 | IHS ~1,618 | HHS umbrella (DHHS/OS) ~964 | CMS ~658 | ASPR ~412.
-- A single funder with ~100% of rows = the blanket CROSS JOIN bug.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.hhs_taggs_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Scope-filter audit: awards excluded by the research/discretionary rule.
-- Expect ~6,383 excluded: OPEN-ENDED ~2,803 (CMS Medicaid-adjacent),
-- BLOCK ~2,344 (SAMHSA block grants), DIRECT PAYMENT ~898 (IHS tribal),
-- CLOSED-ENDED ~338 (CMS).
WITH rolled AS (
    SELECT opdiv, UPPER(TRIM(award_number)) AS award_number,
           MODE(NULLIF(TRIM(award_class_type), '')) AS cls
    FROM openalex.awards.hhs_taggs_raw
    WHERE award_number IS NOT NULL AND TRIM(award_number) != ''
    GROUP BY 1, 2
)
SELECT cls AS excluded_class, COUNT(*) AS awards_dropped
FROM rolled
WHERE cls NOT IN ('DISCRETIONARY', 'COOPERATIVE AGREEMENT')
GROUP BY cls ORDER BY awards_dropped DESC;

In [ ]:
%sql
-- §6.3 Data completeness
-- Expected: 100% title, 0% description (not published), ~62% amount
-- (window-sum > 0), ~64% start_date (budget-period-1 in-window),
-- 0% PI (not published — org-level recipient kept in helper columns).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(recipient_name) as has_recipient,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_dates,
    ROUND(COUNT(recipient_name) * 100.0 / COUNT(*), 1) as pct_recipient
FROM openalex.awards.hhs_taggs_awards;

In [ ]:
%sql
-- §6.7 amount/currency fail-fast check.
-- pct_amount ~62% is BELOW the 50%+ bar but expected and documented: the
-- export window (FY2022-26) sums to <= $0 for continuation awards whose only
-- in-window actions are $0 admin actions or de-obligations. Currency = USD
-- only. avg amount should sit in the millions (multi-year discretionary).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount,
    SUM(amount) AS total_usd
FROM openalex.awards.hhs_taggs_awards;

In [ ]:
%sql
-- Check year distribution (start_year only exists for awards that began
-- in-window, so expect 2021-2026 calendar years and NULL for the rest)
SELECT start_year, COUNT(*) as cnt, SUM(amount) as total_funding
FROM openalex.awards.hhs_taggs_awards
GROUP BY start_year
ORDER BY start_year DESC NULLS LAST;

In [ ]:
%sql
-- §6.4a frequency check (display_name analog; PI is NULL by design).
-- Federal program grants legitimately repeat generic titles across states
-- (e.g. "State Health Insurance Assistance Program") — a long tail with
-- some repeated program titles is expected; one title covering most rows is not.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.hhs_taggs_awards
GROUP BY 1 ORDER BY n DESC
LIMIT 20;

In [ ]:
%sql
-- Check funder_scheme distribution (top 20 assistance listings)
SELECT funder_scheme, COUNT(*) as cnt, SUM(amount) as total_funding
FROM openalex.awards.hhs_taggs_awards
WHERE funder_scheme IS NOT NULL
GROUP BY funder_scheme
ORDER BY cnt DESC
LIMIT 20;

In [ ]:
%sql
-- Check top recipients (org-level; PI not published)
SELECT recipient_name, recipient_state, COUNT(*) as cnt, SUM(amount) as total_funding
FROM openalex.awards.hhs_taggs_awards
WHERE recipient_name IS NOT NULL
GROUP BY recipient_name, recipient_state
ORDER BY total_funding DESC NULLS LAST
LIMIT 20;

In [ ]:
%sql
-- Duplicate-id guard: xxhash64(funder_id:award_number) must be unique
SELECT COUNT(*) AS total, COUNT(DISTINCT id) AS distinct_ids
FROM openalex.awards.hhs_taggs_awards;